## Include Cloud ingestion to the pipeline

In [2]:
!pip install -r ../../requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 506.6 kB/s eta 0:00:0000:0100:02

[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


In [1]:
%load_ext autoreload
%autoreload 2

In [22]:
# Imports
import csv, gzip, os
import io
from urllib.request import Request, urlopen, urlretrieve
import boto3
import psycopg2
from contextlib import closing
import download
import load_local_data
from storage import S3Storage, LocalStorage, get_storage
from pathlib import Path

### Build storage classes

#### local

In [ ]:
# Put existing functions into class
class LocalStorage:
    def __init__(self, root: str):
        self.root = Path(root).expanduser().resolve()

    def find_existing_data(self) -> dict:
        candidates = {}
        for directory in self.root.iterdir():
            if directory.is_dir() and directory.name.startswith("data_"):
                candidates[directory.name] = [f.name for f in directory.iterdir() if f.name.startswith(("listings", "reviews")) and f.name.endswith(".csv.gz")]
        return candidates

    def store_file(self, cohort: str, file_name: str, url: str) -> None:
        target = self.root / cohort
        target.mkdir(parents=True, exist_ok=True)
        urlretrieve(url, target / file_name)

    def read_header(self, cohort: str, file_name: str) -> list[str]:
        with gzip.open(self.root / cohort / file_name, "rt", encoding="utf-8", newline="") as f:
            return next(csv.reader(f))

    def copy_into_tmp(self, cur, cohort: str, file_name: str) -> None:
        with gzip.open(self.root / cohort / file_name, "rt", encoding="utf-8", newline="") as f:
            cur.copy_expert("COPY tmp_load FROM STDIN WITH (FORMAT csv, HEADER true)", f)

In [23]:
local = get_storage("local")

In [27]:
print(local.root, local.root.exists())
print(LocalStorage.find_existing_data)
print(list(local.find_existing_data().keys()))

/data/jakob/Desktop/data_engineering/c_herumspielen/aa_real_dbt_10.-12.26/data True
<function LocalStorage.find_existing_data at 0x7cc24810fb00>
['data_26.06.26', 'data_25.09.23', 'data_25.12.27', 'data_26.09.19', 'data_26.03.28']


In [28]:
local_cohort = sorted(local.find_existing_data())[-1]
local.find_existing_data()[local_cohort]

['listings.csv.gz', 'reviews.csv.gz']

In [29]:
local.read_header(local_cohort, "listings.csv.gz")[:10]

['id',
 'listing_url',
 'scrape_id',
 'last_scraped',
 'source',
 'name',
 'description',
 'neighborhood_overview',
 'picture_url',
 'host_id']

#### cloud

In [59]:
s3 = boto3.client("s3", region_name=os.environ["AWS_REGION"])
bucket = os.environ["S3_BUCKET"]
key = "raw/data_26.09.19/listings.csv.gz"
url = "https://data.insideairbnb.com/germany/be/berlin/2026-09-19/data/listings.csv.gz"
s3, bucket, key

(<botocore.client.S3 at 0x7cc23411aae0>,
 'airbnb-berlin-data-2026-j',
 'raw/data_26.09.19/listings.csv.gz')

In [60]:
def store_file(s3, bucket: str, key: str, url: str) -> None:
    with urlopen(url) as response:
        s3.upload_fileobj(response, bucket, key, ExtraArgs={"ContentEncoding": "gzip", "ContentType": "text/csv"},)

In [48]:
store_file(s3, bucket, key, url)

In [50]:
head = s3.head_object(Bucket=bucket, Key=key)
print(key, head["ContentLength"], head.get("ContentEncoding"), head["LastModified"])

raw/data_26.09.19/listings.csv.gz 7032618 gzip 2026-10-23 13:53:24+00:00


In [52]:
def read_header(s3, bucket: str, key: str) -> list[str]:
    body = s3.get_object(Bucket=bucket, Key=key)["Body"]
    try:
        with gzip.GzipFile(fileobj=body) as gz:
            return next(csv.reader(io.TextIOWrapper(gz, encoding="utf-8", newline="")))
    finally:
        body.close()

In [57]:
header = read_header(s3, bucket, key)
print("columns:\n", header)

columns:
 ['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview', 'picture_url', 'host_id', 'host_url', 'host_profile_id', 'host_profile_url', 'host_name', 'host_since', 'hosts_time_as_user_years', 'hosts_time_as_user_months', 'hosts_time_as_host_years', 'hosts_time_as_host_months', 'host_location', 'host_about', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_thumbnail_url', 'host_picture_url', 'host_neighbourhood', 'host_listings_count', 'host_total_listings_count', 'host_verifications', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms', 'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price', 'price_quote_checkin_date', 'price_quote_checkout_date', 'price_quote_total_price', 'price_quote_price_per_night', 'price_quote_raw', 'm

In [42]:
# New adjusted functions for S3
class S3Storage:
    def __init__(self, bucket: str, region: str, prefix: str = "raw"):
        self.bucket = bucket
        self.region = region
        self.prefix = prefix # folder for raw csv.gz data
        self.s3 = boto3.client("s3", region_name=region)

    def _key(self, cohort: str, file_name: str) -> str:
        return f"{self.prefix}/{cohort}/{file_name}"

    def find_existing_data(self) -> dict:
        candidates = {}
        paginator = self.s3.get_paginator("list_objects_v2")
        for page in paginator.paginate(Bucket=self.bucket, Prefix=f"{self.prefix}/"):
            for obj in page.get("Contents", []):
                parts = obj["Key"].split("/")
                if (len(parts) == 3 and parts[1].startswith("data_") and parts[2].startswith(("listings", "reviews")) and parts[2].endswith(".csv.gz")):
                    candidates.setdefault(parts[1], []).append(parts[2])
        return candidates

    def store_file(self, cohort: str, file_name: str, url: str) -> None:
        with urlopen(url) as response:
            self.s3.upload_fileobj(response, self.bucket, self._key(cohort, file_name), ExtraArgs={"ContentEncoding": "gzip", "ContentType": "text/csv"},)

    def read_header(self, cohort: str, file_name: str) -> list[str]:
        body = self.s3.get_object(Bucket=self.bucket, Key=self._key(cohort, file_name))["Body"]
        try:
            with gzip.GzipFile(fileobj=body) as gz:
                return next(csv.reader(io.TextIOWrapper(gz, encoding="utf-8", newline="")))
        finally:
            body.close()

    def copy_into_tmp(self, cur, cohort: str, file_name: str) -> None:
        cur.execute(
            "SELECT aws_s3.table_import_from_s3('tmp_load', '', '(FORMAT csv, HEADER true)', aws_commons.create_s3_uri(%s, %s, %s))",
            (self.bucket, self._key(cohort, file_name), self.region),
        )

In [31]:
s3s = get_storage("cloud")
print(s3s.bucket, s3s.region, os.getenv("AWS_PROFILE"))

airbnb-berlin-data-2026-j eu-central-1 pipeline


In [34]:
[o["Key"] for o in s3s.s3.list_objects_v2(Bucket=s3s.bucket).get("Contents", [])]

['raw/data_25.12.27/listings.csv.gz',
 'raw/data_25.12.27/reviews.csv.gz',
 'raw/data_26.03.28/listings.csv.gz',
 'raw/data_26.03.28/reviews.csv.gz',
 'raw/data_26.06.26/listings.csv.gz',
 'raw/data_26.06.26/reviews.csv.gz',
 'raw/data_26.09.19/listings.csv.gz',
 'raw/data_26.09.19/reviews.csv.gz']

In [35]:
s3s.find_existing_data()

{'data_25.12.27': ['listings.csv.gz', 'reviews.csv.gz'],
 'data_26.03.28': ['listings.csv.gz', 'reviews.csv.gz'],
 'data_26.06.26': ['listings.csv.gz', 'reviews.csv.gz'],
 'data_26.09.19': ['listings.csv.gz', 'reviews.csv.gz']}

In [36]:
s3s._key(cohort, file_name)

'raw/data_26.09.19/listings.csv.gz'

In [39]:
s3s.read_header(cohort, file_name)[:10]

['id',
 'listing_url',
 'scrape_id',
 'last_scraped',
 'source',
 'name',
 'description',
 'neighborhood_overview',
 'picture_url',
 'host_id']

In [ ]:
download.candidates_for_download(soup, s3s)

{}

In [61]:
load_local_data.require_env("DB_HOST")

'airbnb-berlin-db.czc6auq64zic.eu-central-1.rds.amazonaws.com'

In [62]:
soup = await download.get_existing_online_data()
candidates = download.candidates_for_download(soup, s3s)
cohort = sorted(candidates)[-1]
file_name = "listings.csv.gz"
url = candidates[cohort][file_name]
cohort, url

IndexError: list index out of range

In [63]:
with urlopen(Request(url, method="HEAD")) as r:
    online_size = int(r.headers["Content-Length"])
    print(online_size, r.headers.get("Content-Type"), r.headers.get("Content-Encoding"))

7032618 application/x-gzip None


In [ ]:
s3s.find_existing_data()

{'data_26.09.19': ['listings.csv.gz']}

In [17]:
cloud_header = s3s.read_header(cohort, file_name)
print(len(cloud_header), cloud_header[:100])

if cohort in local.find_existing_data():
    assert cloud_header == local.read_header(cohort, file_name)

90 ['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview', 'picture_url', 'host_id', 'host_url', 'host_profile_id', 'host_profile_url', 'host_name', 'host_since', 'hosts_time_as_user_years', 'hosts_time_as_user_months', 'hosts_time_as_host_years', 'hosts_time_as_host_months', 'host_location', 'host_about', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_thumbnail_url', 'host_picture_url', 'host_neighbourhood', 'host_listings_count', 'host_total_listings_count', 'host_verifications', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms', 'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price', 'price_quote_checkin_date', 'price_quote_checkout_date', 'price_quote_total_price', 'price_quote_price_per_night', 'price_quote_raw', 'minimum_

In [12]:
load_local_data.check_contract(s3s, (cohort,))

In [ ]:
# Check temporary tables in cloud
from psycopg2 import sql

cols = sql.SQL(", ").join(sql.SQL("{} text").format(sql.Identifier(c)) for c in cloud_header)

with closing(psycopg2.connect(load_local_data.DSN)) as conn:
    with conn.cursor() as cur:
        cur.execute(sql.SQL("CREATE TEMP TABLE tmp_load ({})").format(cols))
        s3s.copy_into_tmp(cur, cohort, file_name)
        cur.execute("SELECT COUNT(*) FROM tmp_load")
        print("rows:", cur.fetchone()[0])
        cur.execute("SELECT id, name FROM tmp_load LIMIT 3")
        print(cur.fetchall())
    conn.rollback()

rows: 12376
[('1733333348320050763', "Schöne Souterrain Wohnung unweit vom Ku'damm"), ('811527740012470418', 'Moderne Wohnung in Top-Lage'), ('638045311043310955', 'Berlin - Neukölln - 2 bedroom flat')]


In [ ]:
with closing(psycopg2.connect(load_local_data.DSN)) as conn:
    with conn.cursor() as cur:
        cur.execute(
            "SELECT * FROM aws_s3.query_export_to_s3('SELECT 1 AS test_col', aws_commons.create_s3_uri(%s, %s, %s), options := 'format csv, header true')", (s3s.bucket, "ml/test_export.csv", s3s.region),)
        print(cur.fetchone())

print(s3s.s3.list_objects_v2(Bucket=s3s.bucket, Prefix="ml/").get("KeyCount"))
s3s.s3.delete_object(Bucket=s3s.bucket, Key="ml/test_export.csv")   # cleanup

(1, 1, 11)
1


{'ResponseMetadata': {'RequestId': 'PBQEGFZX7N83G01D',
  'HostId': 'iiB37sdGVbIrpJQcq9GZZ6EkdLWke8iu4Nt+HS2X4EKCzhkbXqBxfhURVw4eMpv6HEN3tc4dR1T6zmrgLHgTb12TOfO/0/W/',
  'HTTPStatusCode': 204,
  'HTTPHeaders': {'x-amz-id-2': 'iiB37sdGVbIrpJQcq9GZZ6EkdLWke8iu4Nt+HS2X4EKCzhkbXqBxfhURVw4eMpv6HEN3tc4dR1T6zmrgLHgTb12TOfO/0/W/',
   'x-amz-request-id': 'PBQEGFZX7N83G01D',
   'server': 'AmazonS3',
   'date': 'Fri, 23 Oct 2026 13:17:55 GMT'},
  'RetryAttempts': 0}}

In [ ]:
remaining = download.candidates_for_download(soup, s3s)
print(remaining.get(cohort))

{'reviews.csv.gz': 'https://data.insideairbnb.com/germany/be/berlin/2026-09-19/data/reviews.csv.gz'}


### Test cloud pipeline ingestion

In [58]:
!python ingestion_pipeline.py --target cloud

Ingestion successful: data_25.12.27, data_26.03.28, data_26.06.26, data_26.09.19


In [66]:
from dotenv import load_dotenv

load_dotenv(override=True)

os.environ.update({
    "PGHOST": os.environ["DB_HOST"],
    "PGDATABASE": os.environ["DB_NAME"],
    "PGUSER": os.environ["DB_USER"],
    "PGPASSWORD": os.environ["DB_PASSWORD"],
    "PGSSLMODE": "require",
})

In [ ]:
!psql -c "SELECT data_cohort, COUNT(*) FROM airbnb_raw_data.listings GROUP BY data_cohort;"

 data_cohort | count 
-------------+-------
 25.12.27    | 14470
 26.03.28    |  5856
 26.06.26    | 12776
 26.09.19    | 12376
(4 rows)



In [70]:
!psql -c "\dx"

                           List of installed extensions
    Name     | Version |   Schema   |                 Description                 
-------------+---------+------------+---------------------------------------------
 aws_commons | 1.2     | public     | Common data types across AWS services
 aws_s3      | 1.2     | public     | AWS S3 extension for importing data from S3
 plpgsql     | 1.0     | pg_catalog | PL/pgSQL procedural language
(3 rows)

